# NB01 — Data Preparation and EDA · Binary Triage Gate (A4)

**Task:** classify `NORMAL` × `ALTERED` (triage gate) in Brazilian upper GI endoscopy.
**Source of truth:** the 5 frozen folds in `splits/` (n=1,984). This notebook **does not train** — it validates the data, defines the binary target and generates characterization figures (EN/PT).

Outputs: figures in `figures/en|pt`, summary in `results/metrics/nb01_eda.json`.


In [ ]:
import sys, io, os
from pathlib import Path
# project root = parent of the notebooks folder
ROOT = Path.cwd() if (Path.cwd()/'configs').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('ROOT =', ROOT)
from configs import config as C
from src import data, utils
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams.update({'savefig.dpi': C.DPI, 'axes.grid': True, 'grid.alpha':0.3})


## 1. Environment / GPU
Confirms if a GPU is available (training runs on GPU; here in NB01 it is light).

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('chosen device:', utils.get_device())
print('IMAGES_DIR:', C.IMAGES_DIR, '| exists:', C.IMAGES_DIR.exists())


## 2. Loading splits and defining the binary target
`y = ALTERED`. The binary target (`y`) was already handled in the splits generation script (`data.py`), under the following rules: policy for ambiguous cases `drop_neither` (removes the 6 "neither NORMAL nor ALTERED" images); images marked as both become ALTERED=1 (clinical conservative).


In [ ]:
rows=[]
for f in range(C.N_FOLDS):
    for part in ['train','val','test']:
        df = data.load_fold_df(f, part)
        rows.append({'fold':f,'part':part,'n':len(df),'pos(ALTERED)':int(df.y.sum()),
                     'prev_%':round(100*df.y.mean(),2)})
folds_df = pd.DataFrame(rows)
display(folds_df)
print('pos_weight per fold:', {f: round(data.compute_pos_weight(f),3) for f in range(C.N_FOLDS)})


## 3. Sanity check: no leakage between train/val/test (each fold)
`image_name` intersection is verified. **NOTE FOR THE ARTICLE:** It is assumed that the dataset contains strictly one image per patient (or that the grouping in `data.py` has already isolated patients). Otherwise, it is necessary to validate the leakage via `patient_id`.

In [ ]:
for f in range(C.N_FOLDS):
    df_tr = data.load_fold_df(f,'train')
    df_va = data.load_fold_df(f,'val')
    df_te = data.load_fold_df(f,'test')
    
    tr=set(df_tr.image_name)
    va=set(df_va.image_name)
    te=set(df_te.image_name)
    print(f'fold {f} (image_name): tr∩te={len(tr&te)} tr∩va={len(tr&va)} va∩te={len(va&te)}  (expected 0)')
    
    # If patient_id exists, also validate leakage at the patient level:
    # if 'patient_id' in df_tr.columns:
    #     tr_p=set(df_tr.patient_id); va_p=set(df_va.patient_id); te_p=set(df_te.patient_id)
    #     print(f'fold {f} (patient_id): tr∩te={len(tr_p&te_p)} tr∩va={len(tr_p&va_p)} va∩te={len(va_p&te_p)}  (expected 0)')


## 4. Dataset characterization (deduplicated, n=1,984)
NORMAL/ALTERED distribution, by center (`LOCAL`) and prevalence of pathologies (context for gate safety analysis).


In [ ]:
# complete deduplicated dataset (we assume the union of fold 0 represents the total dataset)
full = pd.concat([data.load_fold_df(0,p) for p in ['train','val','test']], ignore_index=True)
full = full.drop_duplicates('image_name').reset_index(drop=True)
# Recommendation for the article: make sure len(full) reflects 100% of the data before the ML cut.
assert len(full) > 0, 'Empty dataset'
print('n =', len(full))
print('ALTERED =', int(full.y.sum()), f'({100*full.y.mean():.2f}%)')
print('by center (LOCAL):')
display(full.groupby('LOCAL')['y'].agg(['count','mean']).round(3))
PAT=C.PATHOLOGIES
prev = {L:(int(full[L].sum()), round(100*full[L].mean(),2)) for L in ['NORMAL','ALTERADO']+PAT}
prev_df = pd.DataFrame(prev, index=['positives','prev_%']).T
display(prev_df)


## 5. Characterization figures (EN/PT, saved in figures/)

In [ ]:
# 5a. Target balance by center
for lang in C.LANGS:
    t = ('Target balance by center','Center','Proportion') if lang=='en' else \
        ('Balanço do alvo por centro','Centro','Proporção')
    fig,ax=plt.subplots(figsize=(5,4),constrained_layout=True)
    g=full.groupby('LOCAL')['y'].mean()
    ax.bar([str(i) for i in g.index],[1-v for v in g.values],label=('Normal' if lang=='en' else 'Normal'))
    ax.bar([str(i) for i in g.index],g.values,bottom=[1-v for v in g.values],
           label=('Altered' if lang=='en' else 'Alterado'))
    ax.set(title=t[0],xlabel=t[1],ylabel=t[2]); ax.legend(loc='upper right',fontsize=8)
    out_dir = C.FIG_EN if lang=='en' else C.FIG_PT
    out_dir.mkdir(parents=True,exist_ok=True)
    fig.savefig(out_dir/'eda_target_by_center.png',bbox_inches='tight')
    plt.close(fig)
print('eda_target_by_center saved (en/pt)')

# 5b. Pathology prevalence (log)
for lang in C.LANGS:
    t=('Pathology prevalence (within altered context)','Prevalence (%)') if lang=='en' else \
      ('Prevalência das patologias','Prevalência (%)')
    fig,ax=plt.subplots(figsize=(6,4),constrained_layout=True)
    vals=[100*full[L].mean() for L in C.PATHOLOGIES]
    ax.barh(C.PATHOLOGIES, vals, color='#E45756')
    ax.set(title=t[0],xlabel=t[1]); ax.invert_yaxis()
    for i,v in enumerate(vals): ax.text(v+0.1,i,f'{v:.1f}%',va='center',fontsize=8)
    out_dir = C.FIG_EN if lang=='en' else C.FIG_PT
    fig.savefig(out_dir/'eda_pathology_prevalence.png',bbox_inches='tight')
    plt.close(fig)
print('eda_pathology_prevalence saved (en/pt)')


## 6. Pathology in NORMAL images — basis for the gate safety analysis

In [ ]:
patho_count = full[C.PATHOLOGIES].sum(axis=1)
normal = full[full.y==0]
n_norm_patho = int((normal[C.PATHOLOGIES].sum(axis=1)>0).sum())
print(f'NORMAL images with >=1 labeled pathology: {n_norm_patho} / {len(normal)}')
alt = full[full.y==1]
print(f'ALTERED without any labeled pathology: {int((alt[C.PATHOLOGIES].sum(axis=1)==0).sum())} / {len(alt)}')
print('-> these numbers enter the A4 safety/ambiguity discussion')


## 7. Save JSON summary (consolidates EDA)

In [ ]:
def wilson_ci(p, n, z=1.96):
    if n == 0: return (0.0, 0.0)
    denominator = 1 + z**2/n
    center = (p + z**2 / (2*n)) / denominator
    spread = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / denominator
    return (center - spread, center + spread)

summary = {
  'n_total': int(len(full)),
  'altered_pos': int(full.y.sum()), 
  'altered_prev_pct': round(100*full.y.mean(),2),
  'altered_prev_ci_pct': [round(100*x,2) for x in wilson_ci(full.y.mean(), len(full))],
  'by_center': {str(k):{
          'n':int(v['count']),
          'altered_prev':round(float(v['mean']),3),
          'altered_prev_ci': [round(x,3) for x in wilson_ci(float(v['mean']), int(v['count']))]
      } for k,v in full.groupby('LOCAL')['y'].agg(['count','mean']).iterrows()},
  'pathology_prevalence_pct': {L: {
        'prev': round(100*full[L].mean(),2), 
        'ci': [round(100*x,2) for x in wilson_ci(full[L].mean(), len(full))]
    } for L in C.PATHOLOGIES},
  'normal_with_pathology': n_norm_patho,
  'folds': folds_df.to_dict(orient='records'),
}
utils.save_json(summary, C.METRICS_DIR/'nb01_eda.json')
print('saved: results/metrics/nb01_eda.json')
print('NB01 OK — data validated, binary target defined, EN/PT figures and confidence intervals generated.')
